# R1-28 — Effect Sizes + 95% Confidence Intervals
**SSHO-D-26-05807** · run in Google Colab

Computes effect size + bootstrap 95% CI for all main significance tests. Nothing here changes a published point estimate; it **adds** effect-size + CI reporting. Each point estimate is printed so you can confirm it reproduces the manuscript value **before** trusting its CI.

**Files required** (comma-delimited, upload to working dir):
1. `Data_diffusion_analysis_hashed.csv` — KW cascade depth + Mann-Whitney engagement CI
2. `community_porosity.csv` — Wilcoxon porosity
3. `actor_roles_classification_hashed.csv` — chi-square actor role
4. `pairwise_mannwhitney_engagement.csv` — pairs to CI

CI method: bootstrap, B=10,000, seed=42 (consistent with R1-25).

### Setup

In [2]:
import numpy as np
import pandas as pd
from scipy import stats

RNG  = np.random.default_rng(42)
B    = 10_000
DIFF = "Data_diffusion_analysis2.csv"
PORO = "community_porosity.csv"
ROLE = "actor_roles_classification.csv"
PAIR = "pairwise_mannwhitney_engagement.csv"

def ci(boot, lo=2.5, hi=97.5):
    return np.percentile(boot, lo), np.percentile(boot, hi)

### Test 1 — chi-square: actor `structural_role` x `account_type`
Point estimate should reproduce chi2 = 275.41, df = 4. Effect size = Cramer's V + bootstrap CI.

In [3]:
print("="*66); print("TEST 1  chi-square  role x account_type  -> Cramer's V")
r = pd.read_csv(ROLE)
r["structural_role"] = r["structural_role"].str.strip()
r["account_type"]    = r["account_type"].str.strip()

def cramers_v(tab):
    chi2 = stats.chi2_contingency(tab, correction=False)[0]
    n = tab.values.sum()
    k = min(tab.shape) - 1
    return np.sqrt(chi2 / (n * k))

tab = pd.crosstab(r["account_type"], r["structural_role"])
chi2, p, dof, _ = stats.chi2_contingency(tab, correction=False)
V = cramers_v(tab)
print(f"chi2={chi2:.2f}  df={dof}  p={p:.3e}  Cramer's V={V:.3f}")

rows = r[["account_type","structural_role"]].to_numpy()
boot = np.empty(B)
for i in range(B):
    s = rows[RNG.integers(0, len(rows), len(rows))]
    t = pd.crosstab(pd.Series(s[:,0]), pd.Series(s[:,1]))
    boot[i] = cramers_v(t)
lo, hi = ci(boot)
print(f"Cramer's V 95% CI = [{lo:.3f}, {hi:.3f}]")

TEST 1  chi-square  role x account_type  -> Cramer's V
chi2=275.41  df=4  p=2.172e-58  Cramer's V=0.110
Cramer's V 95% CI = [0.099, 0.121]


### Test 2 — chi-square: unique-dyad cross-community (R1-25)
Contingency already known; V should match 0.039. CI via bootstrap of the 2x2 table.
pre: 965 cross / 5706 within (n=6671) · post: 1248 cross / 5954 within (n=7202).

In [4]:
print("="*66); print("TEST 2  chi-square  unique-dyad (R1-25)  -> Cramer's V")
pre  = [(0,1)]*965  + [(0,0)]*5706
post = [(1,1)]*1248 + [(1,0)]*5954
dyad = np.array(pre + post)

def cramers_v_2x2(a):
    t = pd.crosstab(pd.Series(a[:,0]), pd.Series(a[:,1]))
    chi2 = stats.chi2_contingency(t, correction=False)[0]
    return np.sqrt(chi2 / (len(a) * (min(t.shape)-1)))

V2 = cramers_v_2x2(dyad)
print(f"Cramer's V={V2:.3f}  (should match 0.039)")
boot = np.array([cramers_v_2x2(dyad[RNG.integers(0,len(dyad),len(dyad))]) for _ in range(B)])
lo, hi = ci(boot)
print(f"Cramer's V 95% CI = [{lo:.3f}, {hi:.3f}]")

TEST 2  chi-square  unique-dyad (R1-25)  -> Cramer's V
Cramer's V=0.039  (should match 0.039)
Cramer's V 95% CI = [0.023, 0.056]


### Test 3 — Wilcoxon signed-rank: paired porosity (n=38)
Point estimate should reproduce W=178, p=0.025. Effect size = matched-pairs rank-biserial r + bootstrap CI.

In [5]:
print("="*66); print("TEST 3  Wilcoxon  paired porosity (n=38)  -> rank-biserial r")
por = pd.read_csv(PORO)
wide = por.pivot(index="community", columns="phase", values="porosity")
wide = wide.dropna(subset=["Pre-shift","Post-shift"])
pre_p, post_p = wide["Pre-shift"].to_numpy(), wide["Post-shift"].to_numpy()
print(f"n paired = {len(wide)}  (should be 38)")

def wilcoxon_rbc(a, b):
    d = b - a
    d = d[d != 0]
    if len(d) == 0: return 0.0
    ranks = stats.rankdata(np.abs(d))
    rpos = ranks[d > 0].sum(); rneg = ranks[d < 0].sum()
    return (rpos - rneg) / (rpos + rneg)

W, pW = stats.wilcoxon(pre_p, post_p)
rbc = wilcoxon_rbc(pre_p, post_p)
print(f"W={W:.0f}  p={pW:.3f}  rank-biserial r={rbc:.3f}")

idx = np.arange(len(wide))
boot = np.empty(B)
for i in range(B):
    s = idx[RNG.integers(0, len(idx), len(idx))]
    boot[i] = wilcoxon_rbc(pre_p[s], post_p[s])
lo, hi = ci(boot)
print(f"rank-biserial r 95% CI = [{lo:.3f}, {hi:.3f}]")

TEST 3  Wilcoxon  paired porosity (n=38)  -> rank-biserial r
n paired = 38  (should be 38)
W=178  p=0.025  rank-biserial r=0.435
rank-biserial r 95% CI = [0.072, 0.742]


### Test 4 — Kruskal-Wallis: cascade depth (`diffusion_score`) x 7 narratives
ALL narratives, **no** @barengwarga exclusion (matches published p=0.000388). Effect size = epsilon-squared + bootstrap CI.

In [6]:
# ---------------------------------------------------------------------
# TEST 4 (REVISED) — Kruskal-Wallis: cross-community REACH per user
#   across the 4 focal narratives (matches manuscript H=18.26, p=0.000388)
#   NOT diffusion_score. effect size = epsilon-squared + bootstrap CI.
#
#   Extra files: Data_bert_with_topics.csv, network_nodes_with_community.csv,
#                network_edges.csv
#   Adjust sep=';' / ',' and encoding to match your local copies.
# ---------------------------------------------------------------------
from collections import defaultdict
print("="*66); print("TEST 4  Kruskal-Wallis  cross-community reach x 4 narratives  -> epsilon^2")

narrative_map = {0:'Demo & DPR',1:'Gerakan/Hashtag',2:'Politik & Tuntutan',
    3:'Kekerasan Aparat',4:'Ekonomi Rakyat',5:'Affan Kurniawan',6:'Kekerasan Aparat',
    7:'Kekerasan Aparat',8:'Gerakan/Hashtag',9:'Keamanan & Respons',10:'Demo & DPR',
    11:'Kekerasan Aparat',12:'Demo & DPR',13:'Kekerasan Aparat',14:'Gerakan/Hashtag',
    15:'Gerakan/Hashtag',16:'Ekonomi Rakyat',17:'Keamanan & Respons',18:'Keamanan & Respons'}

df    = pd.read_csv("Data_bert_with_topics2.csv", sep=';', encoding='utf-8-sig')
nodes = pd.read_csv("network_nodes_with_community3.csv", sep=';', encoding='utf-8-sig')
edges = pd.read_csv("network_edges.csv", sep=',', encoding='utf-8-sig')

comm_map = dict(zip(nodes['username'], nodes['community']))
df['narrative']      = df['topic'].map(narrative_map)
df['username_clean'] = df['username'].str.strip().str.lstrip('@').str.lower()
df['community']      = df['username_clean'].map(comm_map)
df_comm = df[df['community'].notna()].copy()

edges['source_comm'] = edges['source'].map(comm_map)
edges['target_comm'] = edges['target'].map(comm_map)
edges['narrative']   = edges['dominant_topic'].map(narrative_map)
edges_valid = edges.dropna(subset=['source_comm','target_comm','narrative'])

def dom(x):
    vc = x.dropna().value_counts()
    return vc.index[0] if len(vc) > 0 else 'unknown'
user_narr = df_comm.groupby('username_clean')['narrative'].agg(dom).to_dict()

ucc = defaultdict(set)
for s, sc, tc in zip(edges_valid['source'], edges_valid['source_comm'], edges_valid['target_comm']):
    if sc != tc:
        ucc[s].add(int(tc))

rows = [(user_narr.get(u,'unknown'), len(ucc.get(u,set()))) for u in df_comm['username_clean'].unique()]
udf  = pd.DataFrame(rows, columns=['narrative','reach'])

sel = ['Affan Kurniawan','Demo & DPR','Kekerasan Aparat','Ekonomi Rakyat']
gvals = [udf[udf['narrative']==n]['reach'].values.astype(float) for n in sel]

def eps2(groups):
    H = stats.kruskal(*groups)[0]
    N = sum(map(len, groups)); k = len(groups)
    return (H - k + 1) / (N - k)

H, pH = stats.kruskal(*gvals)
e2 = eps2(gvals)
print(f"n per group = {[len(g) for g in gvals]}")
print(f"H={H:.3f}  p={pH:.6f}  (should match 0.000388)  epsilon^2={e2:.4f}")

boot = np.empty(B)
for i in range(B):
    rs = [g[RNG.integers(0, len(g), len(g))] for g in gvals]   # stratified per group
    boot[i] = eps2(rs)
lo, hi = np.percentile(boot, [2.5, 97.5])
print(f"epsilon^2 95% CI = [{max(lo,0):.4f}, {hi:.4f}]")

TEST 4  Kruskal-Wallis  cross-community reach x 4 narratives  -> epsilon^2
n per group = [366, 1754, 1462, 744]
H=18.264  p=0.000388  (should match 0.000388)  epsilon^2=0.0035
epsilon^2 95% CI = [0.0009, 0.0092]


### Test 5 — Mann-Whitney pairwise engagement (`view_count`)
`effect_size_r` already in the CSV. This ADDS bootstrap CI for each pair's rank-biserial r. Heaviest cell (21 pairs x B resamples) — drop B to 2,000 for a trial run, then 10,000 for final. Exports `R1-28_mannwhitney_viewcount_CI.csv`.

In [8]:
# ---------------------------------------------------------------------
# TEST 5 (REVISED) — Mann-Whitney pairwise engagement (view_count)
#   effect size = rank-biserial r = 1 - 2U/(n1*n2)  (matches manuscript)
#   ADD bootstrap 95% CI per pair.
#   CORRECT input file: Data_with_community3.csv (sep=';'), NOT the
#   diffusion file. This reproduces the published r (n1=714 for Affan).
#   Sign convention here matches the CSV (narrative_1 - narrative_2),
#   so Affan-vs-X are negative; flip sign at write-up to match the
#   positive r in section 4.2 if desired.
# ---------------------------------------------------------------------
print("="*66); print("TEST 5  Mann-Whitney pairwise (view_count)  -> rank-biserial r + CI")

ENG = pd.read_csv(PAIR)                              # existing pairwise file
ENG = ENG[ENG["metric"] == "view_count"].copy()

src = pd.read_csv("Data_with_community3.csv", sep=';', encoding='utf-8-sig')
src["view_count"] = pd.to_numeric(src["view_count"], errors="coerce")

def r_rb(a, b):
    U = stats.mannwhitneyu(a, b, alternative="two-sided")[0]
    return 1 - (2*U) / (len(a)*len(b))

out = []
for _, row in ENG.iterrows():
    a = src[src["narrative"] == row["narrative_1"]]["view_count"].dropna().values
    b = src[src["narrative"] == row["narrative_2"]]["view_count"].dropna().values
    if len(a) == 0 or len(b) == 0:
        out.append((row["narrative_1"], row["narrative_2"], row.get("effect_size_r"), np.nan, np.nan)); continue
    r_obs = r_rb(a, b)
    boot = np.empty(B)
    for i in range(B):
        sa = a[RNG.integers(0, len(a), len(a))]
        sb = b[RNG.integers(0, len(b), len(b))]
        boot[i] = r_rb(sa, sb)
    lo, hi = ci(boot)
    out.append((row["narrative_1"], row["narrative_2"], round(r_obs,3), round(lo,3), round(hi,3)))

res5 = pd.DataFrame(out, columns=["narrative_1","narrative_2","rank_biserial_r","ci_low","ci_high"])
# sanity: r here must match effect_size_r in the existing file
chk = res5.merge(ENG[["narrative_1","narrative_2","effect_size_r"]], on=["narrative_1","narrative_2"])
chk["match"] = (chk["rank_biserial_r"] - chk["effect_size_r"]).abs() < 0.005
print("rows matching published effect_size_r:", int(chk["match"].sum()), "/", len(chk))
print(res5.to_string(index=False))
res5.to_csv("R1-28_mannwhitney_viewcount_CI.csv", index=False)

TEST 5  Mann-Whitney pairwise (view_count)  -> rank-biserial r + CI
rows matching published effect_size_r: 21 / 21
       narrative_1        narrative_2  rank_biserial_r  ci_low  ci_high
   Affan Kurniawan         Demo & DPR           -0.314  -0.361   -0.266
   Affan Kurniawan     Ekonomi Rakyat           -0.429  -0.476   -0.381
   Affan Kurniawan    Gerakan/Hashtag           -0.381  -0.427   -0.336
   Affan Kurniawan Keamanan & Respons           -0.108  -0.193   -0.020
   Affan Kurniawan   Kekerasan Aparat           -0.394  -0.437   -0.350
   Affan Kurniawan Politik & Tuntutan           -0.273  -0.328   -0.220
        Demo & DPR     Ekonomi Rakyat           -0.112  -0.156   -0.068
        Demo & DPR    Gerakan/Hashtag           -0.051  -0.090   -0.013
        Demo & DPR Keamanan & Respons            0.235   0.159    0.310
        Demo & DPR   Kekerasan Aparat           -0.080  -0.119   -0.042
        Demo & DPR Politik & Tuntutan            0.060   0.014    0.104
    Ekonomi Rakyat   

### Bonferroni derivation (NO-COMPUTE — for the writeup)

In [ ]:
print("="*66); print("Bonferroni derivation")
print("alpha = 0.05 / 105 = 0.000476")
print("105 = 5 engagement metrics x 21 narrative pairs; 21 = C(7,2)")
print("comparison rows in file:", pw.shape[0], "(expect ~105)")

print("\nDONE. Cross-check every point estimate against the manuscript before reporting its CI.")